In [1]:
import pandas as pd
import subprocess
import glob
import io

arquivos = sorted(glob.glob("..\\data\\raw\\interrupcoes-energia-eletrica-*.csv"))
AGENTE = "ELETROPAULO"

In [2]:
dfs = []

for f in arquivos:
    # Lê header para nomes de colunas
    with open(f, encoding='latin-1') as fh:
        header = fh.readline()

    # findstr filtra linhas no nível do OS — não carrega o arquivo inteiro no Python
    result = subprocess.run(
        ["findstr", AGENTE, f],
        capture_output=True, text=True, encoding='latin-1'
    )

    if not result.stdout.strip():
        continue

    chunk = pd.read_csv(
        io.StringIO(header + result.stdout),
        sep=';',
        encoding='latin-1',
        dtype=str,
    )

    # Strip trailing spaces (campo SigAgente tem 20 chars fixos no CSV)
    chunk['SigAgente'] = chunk['SigAgente'].str.strip()
    dfs.append(chunk[chunk['SigAgente'] == AGENTE])

aneel = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"{len(aneel):,} registros encontrados")

2,570,618 registros encontrados


In [3]:
aneel.head()

,DatGeracaoConjuntoDados,IdeConjuntoUnidadeConsumidora,DscConjuntoUnidadeConsumidora,DscAlimentadorSubestacao,DscSubestacaoDistribuicao,NumOrdemInterrupcao,DscTipoInterrupcao,IdeMotivoInterrupcao,DatInicioInterrupcao,DatFimInterrupcao,DscFatoGeradorInterrupcao,NumNivelTensao,NumUnidadeConsumidora,NumConsumidorConjunto,NumAno,NomAgenteRegulado,SigAgente,NumCPFCNPJ
0,2026-04-30,12952,PARELHEIROS,PRE 0111,PRE,5308841-1,Não Programada,0,2018-01-11 18:41:10,2018-01-11 22:15:25,Interna - Nao Programada - Proprias do sistema...,13800,40,35138,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
1,2026-04-30,13000,VITÓRIA,VIT 0107,VIT,5339907-1,Não Programada,0,2018-01-21 22:57:31,2018-01-22 07:47:38,Interna - Nao Programada - Proprias do sistema...,240,1,97131,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
2,2026-04-30,12876,CARAPICUIBA,CPI 0112,CPI,5371620-1,Não Programada,0,2018-01-31 15:48:27,2018-01-31 17:29:48,Interna - Nao Programada - Proprias do sistema...,240,1,121701,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
3,2026-04-30,12928,JUQUITIBA,JUQ 0102,JUQ,5402739-1,Não Programada,0,2018-02-14 00:36:33,2018-02-14 08:18:23,Interna - Nao Programada - Proprias do sistema...,13800,1,17997,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193
4,2026-04-30,12895,CONGONHAS,COG 0106,COG,5438990-1,Não Programada,0,2018-02-27 13:20:44,2018-02-27 15:59:22,Interna - Nao Programada - Proprias do sistema...,240,1,55060,2018,ELETROPAULO METROPOLITANA ELETRICIDADE DE SAO ...,ELETROPAULO,61695227000193


In [4]:
aneel.to_csv("..\\data\\interim\\aneel_eletropaulo.csv", index=False, encoding='utf-8')